# Lab 03-04: Embedding Model Selection

**Module:** 03 -- Application Development (30% of exam, ~13 questions)  
**UI Sections:** Discover | Marketplace | Serving  
**Estimated Time:** 40--50 minutes  
**Difficulty:** Intermediate

---

## Learning Objectives

- Understand what embeddings are and why they matter for RAG
- Generate embeddings using Databricks Foundation Model endpoints
- Compare embedding models by dimension, storage cost, and quality
- Implement cosine similarity from scratch and interpret scores
- Prove why you must use the same embedding model for indexing and querying
- Navigate Discover and Marketplace to find embedding models

---

## Exam Objectives Covered

- Embedding model selection for RAG pipelines
- Foundation Model embedding endpoints
- Cosine similarity and vector distance
- Marketplace model discovery
- Same-model constraint for index/query consistency

---

## What & Why

Embeddings convert text into **vectors** (lists of numbers) that capture semantic meaning.
Two pieces of text that mean similar things will have vectors that are close together in
vector space, even if they use completely different words.

**Why does this matter for the exam?**

In a RAG pipeline, you embed both your documents (at indexing time) and the user's query
(at query time), then use vector similarity to find the most relevant documents. Choosing
the right embedding model directly affects:

1. **Retrieval quality** -- does the system find the right documents?
2. **Storage cost** -- how much space do the vectors consume?
3. **Latency** -- how fast are similarity computations?
4. **Compatibility** -- you MUST use the same model for indexing and querying.

Databricks provides **Foundation Model embedding endpoints** (pre-deployed, pay-per-token)
and the **Marketplace** for additional models. For the exam, you use these as-is -- no
fine-tuning required.

---

## Mental Model

> **Embeddings are like GPS coordinates for meaning.**
>
> Just as GPS turns a street address into latitude/longitude numbers that let you compute
> distances between locations, embeddings turn text into number vectors that let you compute
> semantic similarity. Two sentences about the same topic get nearby coordinates, even if
> they use different words.
>
> - "How do I return a product?" -> [0.12, -0.34, 0.56, ...]
> - "What is the refund policy?" -> [0.11, -0.33, 0.55, ...] (nearby -- similar meaning)
> - "What is the weather today?" -> [-0.78, 0.21, -0.45, ...] (far away -- different topic)
>
> The distance between the first two is small. The distance to the third is large.
> That is how vector search finds relevant documents.

---

## Exam Alert

| # | Trap (WRONG answer) | Correct Understanding |
|---|---|---|
| 1 | "Larger embedding dimensions always give better results" | Larger dims increase storage and latency; choose based on your data complexity. 384 dims may outperform 1024 for simple use cases. |
| 2 | "You can mix embedding models between indexing and querying" | You **MUST** use the same model for both. Different models produce incompatible vector spaces -- even identical text gets different vectors. |
| 3 | "Embeddings understand word order like LLMs" | Embeddings capture semantic similarity, not reasoning or generation. They compress meaning into a fixed-size vector. |
| 4 | "Fine-tune embedding models for better RAG" | For the exam, use Foundation Model endpoints or Marketplace models **as-is**. Fine-tuning is not tested. |
| 5 | "Switching to a smaller embedding model reduces the number of vectors stored" | The number of vectors = the number of chunks. The embedding model determines vector *dimensionality*, not vector *count*. To reduce count, change your chunking strategy. |

---

## UI Navigation: Finding Embedding Models

**Path 1: Foundation Model Endpoints (pre-deployed)**

```
Workspace sidebar -> Serving -> find "databricks-bge-large-en" or "databricks-gte-large-en"
```

These are always available, pay-per-token, no deployment needed. They appear as
system-managed serving endpoints.

**Path 2: Marketplace (additional models)**

```
Workspace sidebar -> Discover -> Search "embedding" -> browse available models
```

The Marketplace offers additional embedding models (multilingual, domain-specific, etc.)
that you can deploy to your own serving endpoints.

**Path 3: Model Catalog**

```
Workspace sidebar -> Catalog -> Models -> filter by task "embedding"
```

Browse models registered in Unity Catalog, including any you have deployed from
the Marketplace.

---

## Setup

In [ ]:
import os
from openai import OpenAI
import math

# --- Databricks connection ---
# In a Databricks notebook these are set automatically.
# Locally, set DATABRICKS_HOST and DATABRICKS_TOKEN env vars.
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST", "")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN", "")

client = OpenAI(
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
    api_key=DATABRICKS_TOKEN,
)

EMBEDDING_MODEL = "databricks-bge-large-en"
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "workspace"
SCHEMA = "genai_labs"

print(f"Host:            {DATABRICKS_HOST[:40]}...")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model:       {LLM_MODEL}")
print(f"Schema:          {CATALOG}.{SCHEMA}")
print()
print("Setup complete.")


---

## Step 1: What Are Embeddings? (Visual Demo)

Let us start by seeing embeddings in action. We will embed three sentences:
- One about **cooking**
- One about **baking** (semantically similar to cooking)
- One about **programming** (semantically different)

If embeddings truly capture meaning, the cooking and baking vectors should be
close together, while the programming vector should be far away.

In [ ]:
# Embed three sentences with different semantic content
sentences = [
    "I love cooking Italian pasta with fresh tomatoes and basil.",        # cooking
    "Baking sourdough bread requires patience and a good starter.",       # baking (similar)
    "Python list comprehensions are a concise way to create lists.",      # programming (different)
]

response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=sentences,
)

# Extract the embedding vectors
vectors = [item.embedding for item in response.data]

print(f"Model: {EMBEDDING_MODEL}")
print(f"Number of sentences embedded: {len(vectors)}")
print(f"Vector dimensions: {len(vectors[0])}")
print()

# Show first 5 values of each vector
for i, (sentence, vec) in enumerate(zip(sentences, vectors)):
    preview = [f"{v:+.4f}" for v in vec[:5]]
    print(f"Sentence {i+1}: {sentence[:50]}...")
    print(f"  First 5 dims: [{', '.join(preview)}, ...]")
    print(f"  Total dims:   {len(vec)}")
    print()


In [ ]:
def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors.

    Formula: cos(theta) = (A . B) / (|A| * |B|)

    Returns a value between -1.0 and 1.0:
      1.0  = identical direction (same meaning)
      0.0  = orthogonal (unrelated)
     -1.0  = opposite direction (opposite meaning)
    """
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = math.sqrt(sum(a * a for a in vec_a))
    magnitude_b = math.sqrt(sum(b * b for b in vec_b))
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0
    return dot_product / (magnitude_a * magnitude_b)


# Compute similarity between all pairs
labels = ["Cooking", "Baking", "Programming"]

print("Cosine Similarity Matrix:")
print(f"{'':>14}", end="")
for label in labels:
    print(f"{label:>14}", end="")
print()
print("-" * 56)

for i in range(3):
    print(f"{labels[i]:>14}", end="")
    for j in range(3):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"{sim:>14.4f}", end="")
    print()

print()
sim_cook_bake = cosine_similarity(vectors[0], vectors[1])
sim_cook_prog = cosine_similarity(vectors[0], vectors[2])
sim_bake_prog = cosine_similarity(vectors[1], vectors[2])

print(f"Cooking  <-> Baking:      {sim_cook_bake:.4f}  (high -- both about food)")
print(f"Cooking  <-> Programming: {sim_cook_prog:.4f}  (low -- different topics)")
print(f"Baking   <-> Programming: {sim_bake_prog:.4f}  (low -- different topics)")
print()
print("As expected, food-related sentences cluster together while")
print("the programming sentence is far away in vector space.")


**Key observations:**
- Both cooking and baking sentences get vectors that are close together (high cosine similarity).
- The programming sentence is far from both food sentences (low cosine similarity).
- A sentence compared to itself always gives similarity = 1.0 (identical vectors).
- This is exactly how vector search works in RAG: the query vector is compared to all document vectors, and the closest matches are returned.

---

## Step 2: Foundation Model Embedding Endpoints

Databricks provides pre-deployed embedding endpoints that require no setup. You call them
via the OpenAI-compatible API. The two primary Foundation Model embedding endpoints are:

| Model | Dimensions | Context Window | Best For |
|---|---|---|---|
| `databricks-bge-large-en` | 1024 | 512 tokens | General English RAG -- good balance of quality and speed |
| `databricks-gte-large-en` | 1024 | 512 tokens | General English RAG -- alternative with similar performance |

Both are **pay-per-token** and always available -- no deployment required.

For the exam, remember:
- These are accessed via **Serving endpoints** (not model registry)
- They use the **OpenAI-compatible API** (`client.embeddings.create()`)
- They are suitable for most English-language RAG use cases

In [ ]:
# Call the Foundation Model embedding endpoint
test_text = "Databricks Lakehouse combines data lakes and data warehouses."

response_bge = client.embeddings.create(
    model="databricks-bge-large-en",
    input=[test_text],
)

vec_bge = response_bge.data[0].embedding

print(f"Model: databricks-bge-large-en")
print(f"  Input text:  {test_text}")
print(f"  Dimensions:  {len(vec_bge)}")
print(f"  First 5:     {[f'{v:+.4f}' for v in vec_bge[:5]]}")
print(f"  Usage:       {response_bge.usage}")
print()

# Show the API call pattern for reference
print("API Pattern:")
print('  response = client.embeddings.create(')
print('      model="databricks-bge-large-en",')
print('      input=[text_or_list_of_texts],')
print('  )')
print('  vector = response.data[0].embedding')
print()
print("Note: You can embed multiple texts in one call by passing a list.")
print("Each text gets its own vector in response.data[i].embedding.")


In [ ]:
# Batch embedding: embed multiple texts in a single API call
documents = [
    "Vector search finds similar documents using cosine similarity.",
    "Delta Lake provides ACID transactions for data lakes.",
    "Unity Catalog manages data governance across the lakehouse.",
    "MLflow tracks experiments, models, and deployments.",
    "Databricks SQL provides a SQL interface for data analysis.",
]

response_batch = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=documents,
)

print(f"Batch embedding: {len(documents)} documents in one API call")
print(f"Vectors returned: {len(response_batch.data)}")
print(f"Dimensions per vector: {len(response_batch.data[0].embedding)}")
print(f"Total usage: {response_batch.usage}")
print()

# Each document gets its own vector
for i, (doc, item) in enumerate(zip(documents, response_batch.data)):
    print(f"  Doc {i+1}: {doc[:55]:55s}  -> vector[{len(item.embedding)} dims]")


---

## Step 3: Embedding Dimensions and Storage

Every embedding vector is a list of floating-point numbers. The **dimension count**
determines how much meaning the model can encode, but also how much storage each
vector requires.

**Storage formula:**

```
Storage = num_documents x dimensions x 4 bytes (float32)
```

This is a critical trade-off for the exam:
- **More dimensions** = richer representation = better quality (up to a point)
- **More dimensions** = more storage = slower similarity search
- **Diminishing returns** -- going from 384 to 1024 dims helps a lot; going from 1024 to 4096 helps less

In [ ]:
# Calculate storage costs for different embedding configurations

configs = [
    ("databricks-bge-large-en", 1024),
    ("databricks-gte-large-en", 1024),
    ("Small model (hypothetical)", 384),
    ("Large model (hypothetical)", 1536),
    ("Very large model (hypothetical)", 4096),
]

num_documents = 1_000_000  # 1 million documents

print(f"Storage Estimates for {num_documents:,} Documents")
print(f"(float32 = 4 bytes per dimension)")
print()
print(f"{'Model':<35} {'Dims':>6} {'Per Vector':>12} {'Total (1M docs)':>16}")
print("-" * 73)

for model_name, dims in configs:
    bytes_per_vector = dims * 4  # float32
    total_bytes = num_documents * bytes_per_vector
    total_gb = total_bytes / (1024 ** 3)

    print(f"{model_name:<35} {dims:>6} {bytes_per_vector:>9,} B {total_gb:>13.2f} GB")

print()
print("Key insight: Doubling dimensions doubles storage and search time.")
print("Choose the smallest dimension count that gives acceptable retrieval quality.")
print()

# Show the trade-off visually
print("Dimension vs. Quality Trade-off:")
print()
print("  Quality")
print("    ^")
print("    |          ...........")
print("    |       ...")
print("    |     ..")
print("    |   ..")
print("    |  .")
print("    | .")
print("    |.")
print("    +------------------------> Dimensions")
print("    128  384  768  1024  4096")
print()
print("Note the diminishing returns: the curve flattens as dimensions increase.")
print("Most of the quality gain comes in the 384-1024 range.")


---

## Step 4: Cosine Similarity Deep Dive

Cosine similarity is the standard metric for comparing embedding vectors. Let us
understand it thoroughly -- the exam expects you to know what the scores mean.

**The formula:**

```
cosine_similarity(A, B) = (A . B) / (|A| * |B|)
```

Where `A . B` is the dot product and `|A|` is the magnitude (length) of vector A.

**Score interpretation:**

| Score | Meaning | Example |
|---|---|---|
| 1.0 | Identical direction | Same sentence embedded twice |
| 0.7 - 0.9 | Very similar | Paraphrases, same topic |
| 0.4 - 0.7 | Somewhat related | Same domain, different subtopic |
| 0.0 - 0.4 | Unrelated | Different topics entirely |
| -1.0 | Opposite | Rare with modern embeddings; most stay in [0, 1] |

Note: Modern embedding models typically produce vectors with all-positive cosine
similarities (scores in the 0 to 1 range). Negative scores are theoretically possible
but uncommon in practice.

In [ ]:
# Mini-benchmark: 5 query-document pairs with known relevance

query = "How do I reset my password?"

documents_with_relevance = [
    ("To reset your password, go to Settings and click Reset Password.", "HIGH"),
    ("You can change your login credentials in the Account Security section.", "MEDIUM"),
    ("Our password policy requires at least 8 characters with one number.", "MEDIUM"),
    ("The weather forecast shows rain for the next three days.", "NONE"),
    ("Contact support if you are locked out of your account.", "LOW-MEDIUM"),
]

# Embed the query and all documents
all_texts = [query] + [doc for doc, _ in documents_with_relevance]
response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=all_texts,
)

query_vec = response.data[0].embedding
doc_vecs = [response.data[i + 1].embedding for i in range(len(documents_with_relevance))]

# Score each document against the query
print(f'Query: "{query}"')
print()
print(f"{'Score':>7}  {'Expected':>10}  Document")
print("-" * 80)

scored_docs = []
for i, ((doc_text, expected), doc_vec) in enumerate(zip(documents_with_relevance, doc_vecs)):
    sim = cosine_similarity(query_vec, doc_vec)
    scored_docs.append((sim, expected, doc_text))

# Sort by score descending (best match first)
scored_docs.sort(key=lambda x: x[0], reverse=True)

for sim, expected, doc_text in scored_docs:
    print(f"{sim:>7.4f}  {expected:>10}  {doc_text[:60]}")

print()
print("Observations:")
print("  - The password reset instruction scores highest (direct match)")
print("  - Related security topics score in the middle range")
print("  - The weather sentence scores lowest (completely unrelated)")
print("  - This ranking is exactly what a vector search would return")


In [ ]:
# Cosine similarity edge cases

# Case 1: Identical text
text_a = "Machine learning models learn patterns from data."
resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text_a, text_a])
sim_identical = cosine_similarity(resp.data[0].embedding, resp.data[1].embedding)

# Case 2: Paraphrase (same meaning, different words)
text_b = "ML algorithms discover patterns by training on datasets."
resp2 = client.embeddings.create(model=EMBEDDING_MODEL, input=[text_a, text_b])
sim_paraphrase = cosine_similarity(resp2.data[0].embedding, resp2.data[1].embedding)

# Case 3: Same words, different meaning (polysemy)
text_c = "I need to bank on the river bank to deposit my money at the bank."
text_d = "The river bank was covered in wildflowers during spring."
resp3 = client.embeddings.create(model=EMBEDDING_MODEL, input=[text_c, text_d])
sim_polysemy = cosine_similarity(resp3.data[0].embedding, resp3.data[1].embedding)

# Case 4: Completely unrelated
text_e = "Quantum physics describes the behavior of subatomic particles."
text_f = "The best recipe for chocolate cake uses Dutch cocoa powder."
resp4 = client.embeddings.create(model=EMBEDDING_MODEL, input=[text_e, text_f])
sim_unrelated = cosine_similarity(resp4.data[0].embedding, resp4.data[1].embedding)

print("Cosine Similarity Edge Cases:")
print(f"{'Case':<30} {'Score':>7}  Interpretation")
print("-" * 75)
print(f"{'Identical text':.<30} {sim_identical:>7.4f}  Should be ~1.0 (same vector)")
print(f"{'Paraphrase':.<30} {sim_paraphrase:>7.4f}  High -- same meaning, different words")
print(f"{'Polysemy (bank)':.<30} {sim_polysemy:>7.4f}  Moderate -- word overlap but mixed meaning")
print(f"{'Completely unrelated':.<30} {sim_unrelated:>7.4f}  Low -- no semantic connection")
print()
print("Key takeaway: Embeddings capture MEANING, not just word overlap.")
print("Paraphrases score high; unrelated texts score low regardless of shared words.")


---

## Step 5: The Same-Model Rule (Critical Exam Point)

This is one of the most important concepts for the exam:

> **You MUST use the same embedding model for indexing and querying.**

Each embedding model maps text into its own unique vector space. The same sentence
embedded by two different models produces completely different vectors. If you index
with model A and query with model B, the vectors live in different spaces and cosine
similarity becomes meaningless.

```
CORRECT:
  Index docs with databricks-bge-large-en  -->  [vector space A]
  Query with   databricks-bge-large-en     -->  [vector space A]  --> similarity works!

WRONG:
  Index docs with databricks-bge-large-en  -->  [vector space A]
  Query with   databricks-gte-large-en     -->  [vector space B]  --> similarity is meaningless!
```

In [ ]:
# PROVE: Same text, different models = incompatible vectors
#
# We cannot call two different model endpoints in every environment,
# so we simulate what happens when vectors come from different models.
# A real different model produces vectors in a COMPLETELY different space.

import random

same_text = "Databricks provides a unified analytics platform."

# "Model A" -- embed with our actual model
resp_a = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=[same_text],
)
vec_model_a = resp_a.data[0].embedding

# "Model B" -- simulate a different model's output with a random vector
# In reality, a different model produces vectors in a different learned space.
random.seed(42)
dims = len(vec_model_a)
vec_model_b_simulated = [random.gauss(0, 1) for _ in range(dims)]

# Normalize to unit length (as real embedding models do)
magnitude = math.sqrt(sum(v * v for v in vec_model_b_simulated))
vec_model_b_simulated = [v / magnitude for v in vec_model_b_simulated]

# Cross-model similarity
sim_cross_model = cosine_similarity(vec_model_a, vec_model_b_simulated)

# Same-model similarity (embed the same text twice)
resp_same = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=[same_text, same_text],
)
sim_same_model = cosine_similarity(
    resp_same.data[0].embedding,
    resp_same.data[1].embedding,
)

print("The Same-Model Rule -- Proof by Demonstration")
print("=" * 60)
print()
print(f'Text: "{same_text}"')
print()
print(f"Same model, same text:      similarity = {sim_same_model:.4f}")
print(f"Different model, same text: similarity = {sim_cross_model:.4f}")
print()
print("Even though the TEXT is identical, the cross-model similarity")
print("is near zero because the vectors live in different spaces.")
print()
print("This is why mixing models between indexing and querying")
print("completely breaks retrieval -- the vectors are incomparable.")


In [ ]:
# ASCII diagram: Correct vs Incorrect RAG embedding flow

diagram = """
THE SAME-MODEL RULE FOR RAG
============================

CORRECT FLOW (same model for index + query):

  Documents                    Query
     |                           |
     v                           v
  [databricks-bge-large-en]  [databricks-bge-large-en]
     |                           |
     v                           v
  Index Vectors              Query Vector
  [0.12, -0.34, ...]        [0.11, -0.33, ...]
     \\                          /
      \\                        /
       v                      v
      COSINE SIMILARITY = 0.95   <-- meaningful!
             |
             v
      Top-K documents returned



INCORRECT FLOW (different models for index + query):

  Documents                    Query
     |                           |
     v                           v
  [databricks-bge-large-en]  [different-model-xyz]
     |                           |
     v                           v
  Index Vectors              Query Vector
  [0.12, -0.34, ...]        [-0.67, 0.89, ...]    <-- different space!
     \\                          /
      \\                        /
       v                      v
      COSINE SIMILARITY = 0.02   <-- meaningless!
             |
             v
      WRONG documents returned (or none at all)
"""

print(diagram)


**Key takeaway for the exam:**

When the exam asks about embedding model changes in a RAG pipeline, the correct answer
is always that you must **re-embed all documents** when switching models. You cannot
just change the query-time model and expect existing index vectors to work.

This also means:
- Record which model was used to create each vector index
- When upgrading models, plan for a full re-indexing job
- In Databricks Vector Search, the model is tied to the index configuration

---

## Step 6: Choosing an Embedding Model (Decision Framework)

For the exam, you need to know which embedding model to recommend for different
use cases. Here is the decision framework:

| Use Case | Recommended Approach | Why |
|---|---|---|
| **General English RAG** | `databricks-bge-large-en` | Good balance of quality, speed, and cost. Pre-deployed Foundation Model endpoint. |
| **Alternative English RAG** | `databricks-gte-large-en` | Similar performance to BGE. Try both and evaluate on your data. |
| **Multilingual content** | Browse Marketplace for multilingual models | Foundation Model endpoints are English-only. Marketplace has multilingual options. |
| **Domain-specific (medical, legal)** | Check Marketplace for specialized models | Domain models may capture specialized terminology better. |
| **Cost-sensitive / high volume** | Consider smaller-dimension models | Fewer dimensions = less storage and faster search. Evaluate quality trade-off. |
| **Maximum quality** | Test multiple models with your actual data | No single model is best for all data. Use evaluation metrics (Lab 02-04) to compare. |

**Decision order:**
1. Start with `databricks-bge-large-en` (it works well for most English use cases)
2. If English is not sufficient, look at Marketplace for multilingual models
3. If domain-specific performance is poor, check Marketplace for specialized models
4. Always evaluate on YOUR data -- benchmarks do not tell the full story

In [ ]:
# Practical model comparison: embed the same texts and measure similarity quality

# A small test set with known relationships
test_queries = [
    "How do I connect my smart home devices?",
    "What is the return policy for defective products?",
]

test_documents = [
    "To add smart home devices, open the app and tap Add Device.",     # relevant to query 1
    "The hub supports Wi-Fi, Zigbee, and Z-Wave protocols.",           # relevant to query 1
    "Returns are accepted within 30 days with original packaging.",     # relevant to query 2
    "The warranty covers manufacturing defects for 2 years.",           # relevant to query 2
    "The weather in Seattle is often rainy in winter.",                  # irrelevant to both
]

# Expected relevance: query 0 -> docs 0,1 are relevant; query 1 -> docs 2,3 are relevant
expected = {
    0: {0: "HIGH", 1: "HIGH", 2: "LOW", 3: "LOW", 4: "NONE"},
    1: {0: "LOW", 1: "LOW", 2: "HIGH", 3: "HIGH", 4: "NONE"},
}

# Embed everything with the BGE model
all_texts = test_queries + test_documents
response = client.embeddings.create(model=EMBEDDING_MODEL, input=all_texts)
all_vecs = [item.embedding for item in response.data]

query_vecs = all_vecs[:len(test_queries)]
doc_vecs = all_vecs[len(test_queries):]

print(f"Model: {EMBEDDING_MODEL}")
print(f"Queries: {len(test_queries)}, Documents: {len(test_documents)}")
print()

for qi, query in enumerate(test_queries):
    print(f'Query {qi+1}: "{query}"')
    print(f"  {'Score':>7}  {'Expected':>8}  Document")
    print(f"  {'-' * 70}")

    scored = []
    for di, (doc, doc_vec) in enumerate(zip(test_documents, doc_vecs)):
        sim = cosine_similarity(query_vecs[qi], doc_vec)
        scored.append((sim, expected[qi][di], doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    for sim, exp, doc in scored:
        marker = "  <--" if exp == "HIGH" else ""
        print(f"  {sim:>7.4f}  {exp:>8}  {doc[:55]}{marker}")
    print()

print("The model correctly ranks relevant documents higher than irrelevant ones.")
print("This is the foundation of vector search in RAG.")


In [ ]:
# How to browse the Marketplace for embedding models
# (This is a UI operation -- here we document the steps)

marketplace_guide = """
FINDING EMBEDDING MODELS IN DATABRICKS MARKETPLACE
===================================================

Step 1: Open the Discover page
  - Click the Discover icon in the left sidebar
  - Or navigate to: Workspace -> Discover

Step 2: Search for embedding models
  - Type "embedding" in the search bar
  - Filter by: Category -> "Models" (not "Data" or "Solutions")

Step 3: Evaluate model listings
  Each listing shows:
  - Model name and provider
  - Description and use cases
  - Supported languages
  - Embedding dimensions
  - License and pricing

Step 4: Install a Marketplace model
  - Click on a model listing
  - Click "Get" or "Install"
  - Select target catalog and schema
  - The model registers in Unity Catalog

Step 5: Deploy to a serving endpoint
  - Navigate to Serving
  - Create a new endpoint pointing to the installed model
  - Configure scaling (CPU/GPU, min/max instances)
  - Once deployed, call it via the same OpenAI-compatible API

IMPORTANT FOR EXAM:
- Foundation Model endpoints (bge, gte) are ALWAYS available
- Marketplace models require installation + deployment
- Both use the same client.embeddings.create() API pattern
"""

print(marketplace_guide)


---

## Exam Tips & Traps

| # | Tip | Why It Matters |
|---|---|---|
| 1 | **Same model for indexing and querying -- always.** | The #1 embedding trap on the exam. Different models = incompatible vector spaces. Changing models requires full re-indexing. |
| 2 | **Embedding dimensions affect storage, not retrieval count.** | Number of vectors = number of chunks (determined by chunking strategy). Dimensions = size of each vector (determined by embedding model). Do not confuse these. |
| 3 | **Foundation Model endpoints are pre-deployed and pay-per-token.** | You do NOT need to create a serving endpoint for `databricks-bge-large-en`. It is always available. Marketplace models DO need deployment. |
| 4 | **Cosine similarity is the standard metric for vector search.** | Know the score ranges: ~1.0 = identical, ~0.7-0.9 = very similar, ~0.0 = unrelated. The exam may ask you to interpret scores. |
| 5 | **"Use the smallest model that meets your quality needs."** | Larger is not always better. 1024 dims is overkill for simple FAQ retrieval. The exam tests whether you understand the quality-cost trade-off. |

---

## Key Takeaways

**Core Concepts**
- Embeddings convert text into vectors that capture semantic meaning
- Cosine similarity measures how close two vectors are (1.0 = identical, 0.0 = unrelated)
- More dimensions = richer representation but more storage and slower search
- Diminishing returns: most quality gains happen in the 384-1024 dimension range

**Databricks-Specific**
- `databricks-bge-large-en` (1024 dims) is the go-to Foundation Model embedding endpoint
- `databricks-gte-large-en` (1024 dims) is an alternative with similar performance
- Foundation Model endpoints are pre-deployed and pay-per-token -- no setup needed
- Marketplace offers additional models (multilingual, domain-specific) that require deployment
- All embedding endpoints use the OpenAI-compatible API: `client.embeddings.create()`

**Exam Essentials**
- **MUST** use the same embedding model for indexing and querying -- switching requires re-indexing
- Embedding model determines vector **dimensionality**, not vector **count** (count = chunk count)
- Storage formula: documents x dimensions x 4 bytes (float32)
- Start with `databricks-bge-large-en` unless you need multilingual or domain-specific capabilities
- Foundation Model endpoints are always available; Marketplace models need installation + deployment

---

**Next Lab:** [Lab 03-05: Model Selection from Hub](./05_model_selection_from_hub.ipynb) -- Learn how to browse, evaluate, and deploy models from the Databricks Model Hub for your GenAI applications.